In [1]:
import pandas as pd
import optuna
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import SMOTE

In [2]:
# Load the dataset
df = pd.read_csv(r'C:\Users\khiew\Downloads\FYP Reduced Dataset.csv')

In [3]:
# Drop diseases with less than 50 instances
disease_counts = df['diseases'].value_counts()
valid_diseases = disease_counts[disease_counts >= 1000].index
df = df[df['diseases'].isin(valid_diseases)]

# Assuming that the target variable is 'diseases' and all other variables are input features
X = df.drop('diseases', axis=1)
y = df['diseases']

# Encode the target variable (diseases) if it's a categorical variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
print("Number of remaining classes in training set:", len(np.unique(y_train)))
print("Number of rows left:", len(df))

Number of remaining classes in training set: 37
Number of rows left: 44748


In [4]:
# Apply SMOTE for class balancing in the training set
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Print the new class distribution after SMOTE
print("Class distribution after SMOTE:")
print(pd.Series(y_train_resampled).value_counts())
print("Number of remaining classes in training set:", len(np.unique(y_train_resampled)))
# Print the number of rows in the resampled training set
print("Number of rows in the resampled training set:", len(X_train_resampled))

Class distribution after SMOTE:
22    1005
27    1005
28    1005
34    1005
6     1005
24    1005
35    1005
12    1005
0     1005
19    1005
18    1005
30    1005
13    1005
5     1005
32    1005
23    1005
16    1005
10    1005
21    1005
33    1005
1     1005
29    1005
4     1005
20    1005
7     1005
31    1005
2     1005
36    1005
8     1005
15    1005
26    1005
9     1005
3     1005
17    1005
25    1005
11    1005
14    1005
Name: count, dtype: int64
Number of remaining classes in training set: 37
Number of rows in the resampled training set: 37185


In [8]:
# Optuna optimization function
def objective(trial):
    # Define the hyperparameters to tune
    n_estimators = trial.suggest_int('n_estimators', 50, 500)
    max_depth = trial.suggest_int('max_depth', 10, 100)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 50)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 50)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None, 0.2, 0.5, 0.8])

    # Create RandomForestClassifier with hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42
    )

    # Perform cross-validation (using 5-fold by default)
    score = cross_val_score(model, X_train_resampled, y_train_resampled, cv=5, scoring='accuracy')
    accuracy = score.mean()

    # Log the hyperparameters and accuracy for this trial
    print(f"Trial {trial.number}: n_estimators={n_estimators}, max_depth={max_depth}, "
          f"min_samples_split={min_samples_split}, min_samples_leaf={min_samples_leaf}, "
          f"max_features={max_features}, Accuracy={accuracy:.4f}")

    return accuracy

In [9]:
# Create Optuna study for optimization with persistent storage
study = optuna.create_study(
    direction='maximize',  # Assuming you are maximizing accuracy
    study_name="randomforest_diseases_symptoms_dropextremelymore1000withSMOTE_HugeExperimental_study", 
    storage=r"sqlite:///C:/Users/khiew/Downloads/randomforest.db", 
    load_if_exists=True  # Load the study if it already exists, to resume from the last trial
)

# Optimize the study with your objective function, you can adjust the n_trials as needed
study.optimize(objective, n_trials=100)
# Print the best trial and hyperparameters
print("\nBest Trial:")
print(study.best_trial)
print("Best Hyperparameters:")
print(study.best_trial.params)

[I 2025-04-22 17:27:15,191] A new study created in RDB with name: randomforest_diseases_symptoms_dropextremelymore1000withSMOTE_HugeExperimental_study
[I 2025-04-22 17:27:42,152] Trial 0 finished with value: 0.6424902514454753 and parameters: {'n_estimators': 321, 'max_depth': 17, 'min_samples_split': 14, 'min_samples_leaf': 22, 'max_features': 0.5}. Best is trial 0 with value: 0.6424902514454753.


Trial 0: n_estimators=321, max_depth=17, min_samples_split=14, min_samples_leaf=22, max_features=0.5, Accuracy=0.6425


[I 2025-04-22 17:28:07,280] Trial 1 finished with value: 0.6853032136614227 and parameters: {'n_estimators': 497, 'max_depth': 47, 'min_samples_split': 29, 'min_samples_leaf': 29, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.6853032136614227.


Trial 1: n_estimators=497, max_depth=47, min_samples_split=29, min_samples_leaf=29, max_features=sqrt, Accuracy=0.6853


[I 2025-04-22 17:28:49,573] Trial 2 finished with value: 0.6686836089821164 and parameters: {'n_estimators': 273, 'max_depth': 99, 'min_samples_split': 9, 'min_samples_leaf': 33, 'max_features': None}. Best is trial 1 with value: 0.6853032136614227.


...

Trial 97: n_estimators=382, max_depth=51, min_samples_split=19, min_samples_leaf=5, max_features=0.8, Accuracy=0.6836


[I 2025-04-22 17:57:22,233] Trial 98 finished with value: 0.6867823046927524 and parameters: {'n_estimators': 331, 'max_depth': 61, 'min_samples_split': 46, 'min_samples_leaf': 7, 'max_features': 'sqrt'}. Best is trial 71 with value: 0.6873739411052844.


Trial 98: n_estimators=331, max_depth=61, min_samples_split=46, min_samples_leaf=7, max_features=sqrt, Accuracy=0.6868


[I 2025-04-22 17:57:36,990] Trial 99 finished with value: 0.6869167675137826 and parameters: {'n_estimators': 290, 'max_depth': 65, 'min_samples_split': 48, 'min_samples_leaf': 10, 'max_features': 'sqrt'}. Best is trial 71 with value: 0.6873739411052844.


Trial 99: n_estimators=290, max_depth=65, min_samples_split=48, min_samples_leaf=10, max_features=sqrt, Accuracy=0.6869

Best Trial:
FrozenTrial(number=71, state=TrialState.COMPLETE, values=[0.6873739411052844], datetime_start=datetime.datetime(2025, 4, 22, 17, 47, 47, 759809), datetime_complete=datetime.datetime(2025, 4, 22, 17, 48, 7, 241078), params={'n_estimators': 352, 'max_depth': 40, 'min_samples_split': 49, 'min_samples_leaf': 3, 'max_features': 0.2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=50, step=1), 'max_depth': IntDistribution(high=100, log=False, low=10, step=1), 'min_samples_split': IntDistribution(high=50, log=False, low=2, step=1), 'min_samples_leaf': IntDistribution(high=50, log=False, low=1, step=1), 'max_features': CategoricalDistribution(choices=('sqrt', 'log2', None, 0.2, 0.5, 0.8))}, trial_id=326, value=None)
Best Hyperparameters:
{'n_estimators': 352, 'max_depth': 40, 'min_sa